# Capa Silver - Dimensión de Promociones

## Librerias

In [1]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Variables

In [2]:
%run ../00-Modulos/n_modulo_variables.ipynb

Servicios:
 Warehouse  = s3a://iceberg/warehouse
 EndPoint   = http://localhost:9000
 Uri        = thrift://localhost:9083
Base de Datos Landing =  iceberg.landing
Base de Datos Bronze  =  iceberg.bronze
Base de Datos Silver  =  iceberg.silver
Base de Datos Gold    =  iceberg.gold


## Spark Session

In [3]:
spark = spark_session_hive("Lakehouse-Iceberg")

## Extracción

In [4]:
df_promotions = spark.read.table(f'{vSchemaSilv}.promotions')
df_dim_promotions = spark.read.table(f'{vSchemaGold}.dim_promotions')

## Transformaciones

In [5]:
# Obtener valor maximo de la llave subrogada
max_sk = df_dim_promotions.select(f.coalesce(f.max(f.col("PROMO_SK")), f.lit(0))).first()[0]

# Cambiar formatos de fecha
df_promotions = df_promotions \
    .withColumn("START_DATE", f.date_format(f.col("START_DATE"), "yyyyMMdd").cast("int")) \
    .withColumnRenamed("START_DATE", "START_DK") \
    .withColumn("EXPIRY_DATE", f.date_format(f.col("EXPIRY_DATE"), "yyyyMMdd").cast("int")) \
    .withColumnRenamed("EXPIRY_DATE", "EXPIRY_DK")

# Obtener registros nuevos y calcula su llave subrogada
df_promotions_ins = df_promotions.alias("p") \
    .join(df_dim_promotions.alias("dp"), (f.col("p.PROMO_ID") == f.col("dp.PROMO_ID")), "left_anti") \
    .withColumn("PROMO_SK", max_sk + f.row_number().over(Window.orderBy(f.col("PROMO_ID")))) \
    .select(
        f.col("PROMO_SK"),
        f.col("p.*")
    )

# Obtener registros a actualizar
df_promotions_upd = df_promotions.alias("p") \
    .join(df_dim_promotions.alias("dp"), (f.col("p.PROMO_ID") == f.col("dp.PROMO_ID")), "inner") \
    .select(
        f.col("dp.PROMO_SK"),
        f.col("p.*")
    )

# Seleccion de columnas finales
df_source = df_promotions_ins.unionByName(df_promotions_upd)

## Carga

In [6]:
# Obtenemos variables para merge
target_table = f'{vSchemaGold}.dim_promotions'
join_columns = ["PROMO_SK"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)

## Cierre de SparkSession

In [7]:
stop_session(spark)

Sesión de Spark cerrada correctamente.
